# 🧠 Advanced Data Augmentation with Keras

> **Module:** 03 — Deep Learning with Keras and TensorFlow  
> **Topic:** Data augmentation, normalisation, custom preprocessing  
> **Dataset:** CIFAR-10 (60,000 colour images, 10 classes)

---

## 📋 Overview

In this notebook I implement and experiment with **data augmentation** — a family of techniques that artificially expand a training dataset by applying controlled transformations to existing images. The goal is to make a model more robust by exposing it to variations it might see at inference time.

**What I build:**

| Part | Technique | Tool |
|---|---|---|
| Part 1 | Basic geometric augmentation | `ImageDataGenerator` |
| Part 2 | Feature-wise & sample-wise normalisation | `ImageDataGenerator.fit()` |
| Part 3 | Custom augmentation function (random noise) | `preprocessing_function=` |
| Part 4 | Visualise augmented images | `matplotlib` |
| Exercises | Apply all three techniques to a real image batch | Solved below |

**Learning goals:**
- Implement geometric augmentations (rotation, shift, shear, zoom, flip)
- Understand the difference between feature-wise and sample-wise normalisation
- Write a custom `preprocessing_function` that adds Gaussian noise
- Visualise the effect of each augmentation type on real images

## 🧩 Theory

### Why augment?

A model trained on a small fixed dataset memorises the exact pixel patterns it saw — it [[overfitting|overfits]]. Data augmentation addresses this by generating **new training samples on-the-fly** from the existing ones, so the model never sees the same exact image twice.

**Telecom / RF analogy 📡:** Think of a channel equalizer trained on a limited set of channel snapshots. If you only train on calm, stationary channels, the equalizer collapses when it encounters multipath fading or Doppler shifts. Data augmentation is like synthetically generating diverse channel conditions during training — rotation ≈ phase rotation, zoom ≈ delay spread variation, noise injection ≈ AWGN — so the equalizer generalises to real-world conditions.

### Geometric augmentations

Each transformation applies a spatial mapping $T: \mathbb{R}^{H \times W} \rightarrow \mathbb{R}^{H \times W}$:

| Augmentation | Operation | Formula |
|---|---|---|
| Rotation by angle $\theta$ | Rotate pixel grid | $\begin{pmatrix}x'\\\ y'\end{pmatrix} = \begin{pmatrix}\cos\theta & -\sin\theta \\ \sin\theta & \cos\theta\end{pmatrix}\begin{pmatrix}x\\y\end{pmatrix}$ |
| Width / height shift by fraction $s$ | Translate | $x' = x + s \cdot W$ |
| Shear by angle $\phi$ | Shear mapping | $x' = x + y\tan\phi$ |
| Zoom by factor $z$ | Scale | $x' = x/z,\ y' = y/z$ |
| Horizontal flip | Mirror | $x' = W - 1 - x$ |

### Normalisation

**Feature-wise normalisation** computes statistics across the *entire dataset* and applies them to each sample:

$$x'_{i} = \frac{x_i - \mu_{\text{dataset}}}{\sigma_{\text{dataset}}}$$

**Sample-wise normalisation** computes statistics per *individual sample*:

$$x'_{i} = \frac{x_i - \mu_{\text{sample}}}{\sigma_{\text{sample}}}$$

Feature-wise is like calibrating an RF receiver against a known noise floor measured across many frames. Sample-wise is like AGC (Automatic Gain Control) — normalising each received burst independently regardless of the global signal environment.

### Gaussian noise injection

Adding noise $\epsilon \sim \mathcal{N}(0, \sigma^2)$ to each pixel:

$$x'_i = x_i + \epsilon_i, \quad \epsilon_i \sim \mathcal{N}(0, \sigma^2)$$

This is directly analogous to AWGN (Additive White Gaussian Noise) in communications — the model must learn to recognise objects even through channel noise, improving robustness.

## ⚙️ Part 0 — Setup & Dataset

I start by installing the required libraries, importing everything, and loading CIFAR-10. I normalise pixel values to $[0, 1]$ by dividing by 255 — this is standard preprocessing before augmentation so all operations happen in a consistent numerical range.

In [ ]:
# Install required libraries
!pip install tensorflow==2.16.2 matplotlib==3.9.1 scipy --quiet

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.preprocessing.image import ImageDataGenerator

print(f"TensorFlow version: {tf.__version__}")

# Load CIFAR-10 — 50,000 training images, 10,000 test images
# 10 classes: airplane, automobile, bird, cat, deer, dog, frog, horse, ship, truck
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()

# Normalise pixel values from [0, 255] to [0.0, 1.0]
# x' = x / 255.0
x_train = x_train.astype('float32') / 255.0
x_test  = x_test.astype('float32')  / 255.0

print(f"Training set:  {x_train.shape}  ({x_train.dtype})")
print(f"Test set:      {x_test.shape}   ({x_test.dtype})")
print(f"Pixel range:   [{x_train.min():.2f}, {x_train.max():.2f}]")

I display a 4×4 grid of raw CIFAR-10 training images to get a feel for the data before any augmentation is applied.

In [ ]:
CLASS_NAMES = ['airplane','automobile','bird','cat','deer',
               'dog','frog','horse','ship','truck']

plt.figure(figsize=(10, 10))
for i in range(16):
    plt.subplot(4, 4, i + 1)
    plt.imshow(x_train[i])
    plt.title(CLASS_NAMES[y_train[i][0]], fontsize=8)
    plt.axis('off')
plt.suptitle('📥 CIFAR-10 — Raw Training Images', fontsize=14)
plt.tight_layout()
plt.show()

### 🖼️ Creating a Synthetic Sample Image

For the augmentation demonstrations I also create a simple synthetic image using PIL — a white background with a red square. This makes the geometric transformations visually obvious (it's easy to see rotation, shear, and zoom on a simple shape).

In [ ]:
from PIL import Image, ImageDraw

# Create a 224×224 white image with a red square centre
image = Image.new('RGB', (224, 224), color=(255, 255, 255))
draw  = ImageDraw.Draw(image)
draw.rectangle([(50, 50), (174, 174)], fill=(255, 0, 0))
image.save('sample.jpg')

plt.imshow(image)
plt.title('🖼️ Synthetic sample image (red square on white)')
plt.axis('off')
plt.show()
print("sample.jpg saved.")

In [ ]:
from tensorflow.keras.preprocessing.image import load_img, img_to_array

# Load the sample image and expand to batch format: (H, W, C) → (1, H, W, C)
img_path = 'sample.jpg'
img = load_img(img_path)
x   = img_to_array(img)          # shape: (224, 224, 3)
x   = np.expand_dims(x, axis=0)  # shape: (1, 224, 224, 3)

print(f"Image tensor shape: {x.shape}  (batch=1, H=224, W=224, C=3)")

## 🔄 Part 1 — Basic Geometric Augmentation

`ImageDataGenerator` applies stochastic geometric transformations to images at training time. Each time an image is drawn from the generator, a new random transformation is sampled from within the specified ranges.

The parameters I use:

| Parameter | Value | Effect |
|---|---|---|
| `rotation_range` | 40° | Rotate image by random angle in $[-40°, +40°]$ |
| `width_shift_range` | 0.2 | Shift image left/right by up to $20\%$ of width |
| `height_shift_range` | 0.2 | Shift image up/down by up to $20\%$ of height |
| `shear_range` | 0.2 | Apply shear transformation up to $0.2$ radians |
| `zoom_range` | 0.2 | Zoom in/out by factor in $[0.8, 1.2]$ |
| `horizontal_flip` | True | Randomly mirror the image horizontally |
| `fill_mode` | 'nearest' | Fill newly created pixels with nearest neighbour |

The `fill_mode='nearest'` parameter determines how pixels outside the original image boundary are filled when the image is rotated or shifted. Using `'nearest'` replicates the nearest border pixel — analogous to zero-order hold in signal interpolation.

In [ ]:
# Create ImageDataGenerator with geometric augmentations
datagen_basic = ImageDataGenerator(
    rotation_range=40,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

# Generate 4 augmented versions of the sample image
fig, axes = plt.subplots(1, 4, figsize=(14, 4))
for i, batch in enumerate(datagen_basic.flow(x, batch_size=1)):
    axes[i].imshow(batch[0].astype('uint8'))
    axes[i].set_title(f'🔄 Augmented {i+1}')
    axes[i].axis('off')
    if i >= 3:
        break
plt.suptitle('Part 1 — Basic Geometric Augmentations (rotation, shift, shear, zoom, flip)', fontsize=12)
plt.tight_layout()
plt.show()

Each call to `datagen_basic.flow()` samples fresh random parameters from the configured ranges. The red square appears rotated, shifted, sheared, and/or mirrored — the model seeing these variations during training learns that the *concept* of the object is invariant to those transformations.

## 📐 Part 2 — Feature-wise & Sample-wise Normalisation

Normalisation adjusts the pixel distribution to stabilise training. Two strategies:

**Feature-wise (global):** compute $\mu$ and $\sigma$ across the full dataset, then apply:
$$x'_{c,h,w} = \frac{x_{c,h,w} - \mu_c}{\sigma_c}$$

Here $c$ is the colour channel. This is equivalent to whitening the dataset — each channel is centred and scaled to unit variance globally. Requires a `datagen.fit(data)` call first.

**Sample-wise (local):** compute $\mu$ and $\sigma$ per individual image:
$$x'_{i} = \frac{x_i - \mu_{\text{sample}}}{\sigma_{\text{sample}}}$$

This is like per-burst AGC — normalise each sample independently, regardless of global dataset statistics.

Both can be combined (as below) — feature-wise normalises the global distribution, then sample-wise ensures each individual sample has zero mean and unit variance.

In [ ]:
# Create ImageDataGenerator with both normalisation modes
datagen_norm = ImageDataGenerator(
    featurewise_center=True,           # subtract dataset mean per channel
    featurewise_std_normalization=True, # divide by dataset std per channel
    samplewise_center=True,            # subtract per-image mean
    samplewise_std_normalization=True  # divide by per-image std
)

# fit() computes the dataset-level statistics needed for featurewise normalisation
datagen_norm.fit(x)

# Generate 4 normalised images
fig, axes = plt.subplots(1, 4, figsize=(14, 4))
for i, batch in enumerate(datagen_norm.flow(x, batch_size=1)):
    display_img = np.clip(batch[0], 0, 255).astype('uint8')
    axes[i].imshow(display_img)
    axes[i].set_title(f'📐 Normalised {i+1}')
    axes[i].axis('off')
    if i >= 3:
        break
plt.suptitle('Part 2 — Feature-wise & Sample-wise Normalisation', fontsize=12)
plt.tight_layout()
plt.show()

> **Note on display:** After normalisation, pixel values are centred around 0 with both positive and negative values. When casting to `uint8` for display, values are clipped to $[0, 255]$. The visual appearance may look washed out — this is expected and correct; the network sees the normalised values, not the display-clipped version.

## ⚡ Part 3 — Custom Augmentation: Gaussian Noise

I define a custom preprocessing function that adds Gaussian noise to each image. This is passed via the `preprocessing_function=` argument of `ImageDataGenerator` — Keras calls this function on every batch before returning it.

The noise model:

$$x'_i = x_i + \epsilon_i, \quad \epsilon_i \sim \mathcal{N}(0, \sigma^2)$$

With $\sigma = 0.1$ (i.e., noise standard deviation = 10% of pixel range). This is the classic AWGN model from communications — every pixel receives an independent zero-mean Gaussian perturbation. Training on noisy images forces the network to learn robust, denoising-aware representations.

In [ ]:
def add_random_noise(image):
    """Add AWGN to an image: x' = x + N(0, 0.1²)"""
    noise = np.random.normal(loc=0.0, scale=0.1, size=image.shape)
    return image + noise


# Create ImageDataGenerator with the custom noise function
datagen_noise = ImageDataGenerator(preprocessing_function=add_random_noise)

# Generate 4 noisy versions
fig, axes = plt.subplots(1, 4, figsize=(14, 4))
for i, batch in enumerate(datagen_noise.flow(x, batch_size=1)):
    display_img = np.clip(batch[0], 0, 255).astype('uint8')
    axes[i].imshow(display_img)
    axes[i].set_title(f'⚡ Noisy {i+1}')
    axes[i].axis('off')
    if i >= 3:
        break
plt.suptitle('Part 3 — Custom Augmentation: Gaussian Noise (σ=0.1)', fontsize=12)
plt.tight_layout()
plt.show()

## 📊 Part 4 — Visualise All Augmented Versions

I now generate a 2×2 grid comparing 4 augmented samples using the noise generator. This is the standard visualisation step — showing that each draw from the generator produces a *different* stochastic transformation of the same original image.

In [ ]:
# Visualise 4 noisy augmented versions in a 2×2 grid
plt.figure(figsize=(8, 8))
for i, batch in enumerate(datagen_noise.flow(x, batch_size=1)):
    if i >= 4:
        break
    plt.subplot(2, 2, i + 1)
    plt.imshow(np.clip(batch[0], 0, 255).astype('uint8'))
    plt.title(f'🧪 Version {i+1}')
    plt.axis('off')
plt.suptitle('📊 Visualising Augmented Image Variants (Gaussian Noise)', fontsize=13)
plt.tight_layout()
plt.show()

---

## 🔢 Exercises — Solved

The following exercises apply all three augmentation techniques to a realistic multi-image dataset. I download a set of sample training images and demonstrate each technique on the full batch.

### Exercise 1 — Apply & Visualise Different Augmentation Techniques

**Objective:** Apply geometric augmentations to a batch of real training images and visualise the results.

I download the sample image archive and extract it, then load 3 images as a batch, apply the full `ImageDataGenerator` pipeline, and display 4 augmented outputs.

In [ ]:
# Download and extract sample images
!wget -q https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/RgP3JFNtPTZA34UmG3KZaA/sample-images.zip
!unzip -q sample-images.zip

In [ ]:
from tensorflow.keras.preprocessing.image import array_to_img

# --- Exercise 1 Solution ---

# 1. Define geometric augmentation parameters
datagen_ex1 = ImageDataGenerator(
    rotation_range=40,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

# 2. Load 3 sample images and build a batch
image_paths = [
    'sample_images/training_images1.jpg',
    'sample_images/training_images2.jpg',
    'sample_images/training_images3.jpg'
]

training_images = []
for image_path in image_paths:
    img = load_img(image_path, target_size=(224, 224))
    img_array = img_to_array(img)
    training_images.append(img_array)
training_images = np.array(training_images)  # shape: (3, 224, 224, 3)

print(f"Training batch shape: {training_images.shape}")

# 3. Generate and visualise 4 augmented outputs
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for i, batch in enumerate(datagen_ex1.flow(training_images, batch_size=1)):
    axes[i].imshow(array_to_img(batch[0]))
    axes[i].set_title(f'🔄 Augmented {i+1}')
    axes[i].axis('off')
    if i >= 3:
        break
plt.suptitle('✅ Exercise 1 — Geometric Augmentation on Real Images', fontsize=13)
plt.tight_layout()
plt.show()

### Exercise 2 — Feature-wise & Sample-wise Normalisation

**Objective:** Apply both normalisation modes to the training image batch and visualise normalised outputs.

Key difference from Part 2: I call `datagen.fit(training_images)` on the real 3-image batch so feature-wise statistics ($\mu$, $\sigma$) are computed from that specific dataset.

In [ ]:
# --- Exercise 2 Solution ---

# 1. Create ImageDataGenerator with both normalisation modes
datagen_ex2 = ImageDataGenerator(
    featurewise_center=True,
    featurewise_std_normalization=True,
    samplewise_center=True,
    samplewise_std_normalization=True
)

# 2. Fit on the training batch — computes per-channel mean and std
datagen_ex2.fit(training_images)

print(f"Feature-wise mean (per channel): {datagen_ex2.mean}")
print(f"Feature-wise std  (per channel): {datagen_ex2.std}")

# 3. Generate and visualise 4 normalised images
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for i, batch in enumerate(datagen_ex2.flow(training_images, batch_size=1)):
    display_img = np.clip(array_to_img(batch[0]), 0, 255)
    axes[i].imshow(display_img)
    axes[i].set_title(f'📐 Normalised {i+1}')
    axes[i].axis('off')
    if i >= 3:
        break
plt.suptitle('✅ Exercise 2 — Feature-wise & Sample-wise Normalisation', fontsize=13)
plt.tight_layout()
plt.show()

### Exercise 3 — Custom Augmentation: Add Random Noise

**Objective:** Define and apply a custom noise augmentation function to the real training image batch.

I reuse the `add_random_noise` function from Part 3 and apply it to the multi-image batch.

In [ ]:
# --- Exercise 3 Solution ---

# 1. Custom noise augmentation function (AWGN model: x' = x + N(0, σ²))
def add_random_noise(image):
    """Inject Gaussian noise: x' = x + ε, ε ~ N(0, 0.1²)"""
    noise = np.random.normal(loc=0.0, scale=0.1, size=image.shape)
    return image + noise


# 2. Create ImageDataGenerator with the custom preprocessing function
datagen_ex3 = ImageDataGenerator(preprocessing_function=add_random_noise)

# 3. Generate and visualise 4 noisy images
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for i, batch in enumerate(datagen_ex3.flow(training_images, batch_size=1)):
    axes[i].imshow(np.clip(array_to_img(batch[0]), 0, 255))
    axes[i].set_title(f'⚡ Noisy {i+1}')
    axes[i].axis('off')
    if i >= 3:
        break
plt.suptitle('✅ Exercise 3 — Custom Augmentation: Gaussian Noise on Real Images', fontsize=13)
plt.tight_layout()
plt.show()

---

## 📊 Summary

| Technique | Class / Parameter | When to use | Key formula |
|---|---|---|---|
| **Geometric augmentation** | `ImageDataGenerator(rotation_range=..., zoom_range=..., ...)` | Small datasets, spatial invariance needed | $T: (x,y) \rightarrow (x', y')$ via rotation/shear/scale matrices |
| **Feature-wise normalisation** | `featurewise_center=True` + `datagen.fit(X)` | Global dataset standardisation | $x' = (x - \mu_{\text{dataset}}) / \sigma_{\text{dataset}}$ |
| **Sample-wise normalisation** | `samplewise_center=True` | Per-image AGC-style normalisation | $x' = (x - \mu_{\text{sample}}) / \sigma_{\text{sample}}$ |
| **Custom noise injection** | `preprocessing_function=add_random_noise` | Robustness to sensor noise | $x' = x + \mathcal{N}(0, \sigma^2)$ |

### Key takeaways

- `ImageDataGenerator` applies transformations **on-the-fly** during training — no disk space overhead, infinite variation per epoch
- `featurewise_*` options require calling `datagen.fit(training_data)` first; `samplewise_*` options do not
- `preprocessing_function` is called *after* all built-in augmentations and *before* returning the batch — it's the hook for any custom operation
- Augmentation is applied **only during training**, not during evaluation — the `flow()` generator used for validation should have no augmentation
- The telecom parallel: geometric augmentations ≈ channel diversity training; noise injection ≈ AWGN robustness; normalisation ≈ AGC / whitening pre-filters

### Augmentation as regularisation

Data augmentation is a form of **implicit regularisation**. It reduces overfitting not by adding a penalty term to the loss, but by ensuring the model never sees the exact same image twice:

$$\mathcal{L}_{\text{eff}} = \mathbb{E}_{T \sim p(T)}\left[\mathcal{L}(f(T(x)), y)\right]$$

The model is trained to minimise loss in expectation over the distribution of transformations $p(T)$, which forces it to learn transformation-invariant features.

---

## 🧪 Sandbox

Free experimentation space — try combinations and variations.

In [ ]:
# SANDBOX 1: Combined augmentation — all techniques at once
def add_gaussian_noise(image, sigma=0.05):
    noise = np.random.normal(0, sigma, image.shape)
    return np.clip(image + noise, 0, 255)

datagen_combined = ImageDataGenerator(
    rotation_range=30,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.15,
    horizontal_flip=True,
    fill_mode='reflect',
    preprocessing_function=add_gaussian_noise
)

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for i, batch in enumerate(datagen_combined.flow(x, batch_size=1)):
    row, col = divmod(i, 4)
    axes[row][col].imshow(np.clip(batch[0], 0, 255).astype('uint8'))
    axes[row][col].set_title(f'Combined {i+1}')
    axes[row][col].axis('off')
    if i >= 7:
        break
plt.suptitle('🧪 Sandbox — Geometric + Gaussian Noise Combined', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# SANDBOX 2: IQ-analogy noise
def iq_noise_augmentation(image):
    sigma_iq = 15.0
    sigma_b  = 5.0
    noise_I = np.random.normal(0, sigma_iq, image[:, :, 0].shape)
    noise_Q = np.random.normal(0, sigma_iq, image[:, :, 1].shape)
    noise_B = np.random.normal(0, sigma_b,  image[:, :, 2].shape)
    image_noisy = image.copy()
    image_noisy[:, :, 0] += noise_I
    image_noisy[:, :, 1] += noise_Q
    image_noisy[:, :, 2] += noise_B
    return image_noisy

datagen_iq = ImageDataGenerator(preprocessing_function=iq_noise_augmentation)

fig, axes = plt.subplots(1, 4, figsize=(14, 4))
for i, batch in enumerate(datagen_iq.flow(x, batch_size=1)):
    axes[i].imshow(np.clip(batch[0], 0, 255).astype('uint8'))
    axes[i].set_title(f'📡 IQ-Noise {i+1}')
    axes[i].axis('off')
    if i >= 3:
        break
plt.suptitle('🧪 Sandbox — IQ Channel Noise (R=I, G=Q, B=carrier)', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# SANDBOX 3: Compare augmentation effect on CIFAR-10 images
cifar_batch = x_train[:4] * 255.0

datagen_cifar = ImageDataGenerator(
    rotation_range=20,
    horizontal_flip=True,
    zoom_range=0.1,
    width_shift_range=0.1,
    fill_mode='nearest'
)

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for j in range(4):
    axes[0][j].imshow(x_train[j])
    axes[0][j].set_title(f'Original: {CLASS_NAMES[y_train[j][0]]}')
    axes[0][j].axis('off')
for i, batch in enumerate(datagen_cifar.flow(cifar_batch, batch_size=4)):
    for j in range(4):
        axes[1][j].imshow(np.clip(batch[j], 0, 255).astype('uint8'))
        axes[1][j].set_title('Augmented')
        axes[1][j].axis('off')
    break
plt.suptitle('🧪 Sandbox — Original vs. Augmented CIFAR-10 Images', fontsize=13)
plt.tight_layout()
plt.show()